# SNAP Retailers — Exploration

Every view here comes through `snapshot(as_of)`, so the current map and any
historical year share one taxonomy and one definition of "active".

Run `python src/pipeline.py` first to build `data/snap.duckdb`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from config import connect
from snapshot import snapshot, series

pd.set_option("display.max_rows", 60)
cur = snapshot("2025-12-31")
print(f"{len(cur):,} active retailers")
cur.head()

## Axis 1 — format

In [ ]:
cur.groupby("format").size().sort_values(ascending=False).to_frame("stores")

## Axis 2 — ownership, crossed with format

This is why the axes are kept separate. USDA's "Large Grocery Store" is
overwhelmingly *independent*, not chain supermarkets — a distinction that is
invisible if you collapse size and ownership into one category list.

In [ ]:
ct = pd.crosstab(cur["format"], cur["ownership"])
ct["total"] = ct.sum(axis=1)
ct.sort_values("total", ascending=False)

## Independent grocers only — a candidate map layer

In [ ]:
indie = cur[
    cur["format"].str.startswith("Grocery")
    & cur["ownership"].eq("independent")
    & ~cur["geocode_missing"]
]
print(f"{len(indie):,} independent grocers with coordinates")
indie.groupby("state").size().sort_values(ascending=False).head(15)

## The 20-year history

Note the Small Grocery Store collapse and the Dollar Store rise. Part of the
grocery decline is definitional: FNS tightened stocking standards around
2016-2018, reclassifying stores as well as removing them. Annotate this on any
published chart.

In [ ]:
by_year = series(2006, 2025, by="format")
hist = pd.DataFrame(by_year).T.fillna(0).astype(int)
hist[["Supermarket", "Super Store", "Grocery (Large)", "Grocery (Medium)",
      "Grocery (Small)", "Convenience Store", "Dollar Store",
      "Combination Grocery/Other"]]

In [ ]:
ax = hist[["Grocery (Small)", "Grocery (Medium)", "Grocery (Large)",
           "Supermarket", "Dollar Store"]].plot(figsize=(11, 5))
ax.set_title("SNAP-authorized retailers by format, 2006-2025")
ax.set_ylabel("stores")
ax.axvspan(2016, 2018, alpha=0.12, color="red")
ax.annotate("stocking-standard\nchanges", xy=(2017, ax.get_ylim()[1] * 0.8),
            ha="center", fontsize=8)

## Export a map-ready layer

Drop rows without coordinates (0.65% of stores) before mapping.

In [ ]:
cols = ["record_id", "store_name", "format", "ownership", "brand",
        "chain_category", "city", "state", "county", "zip_code",
        "latitude", "longitude"]
mappable = cur.loc[~cur["geocode_missing"], cols]
mappable.to_csv("../data/map_current.csv", index=False)
print(f"wrote {len(mappable):,} rows to data/map_current.csv")